In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error
from scipy.stats import spearmanr

# ----------------------------------------
# 0. Reload clean data
# ----------------------------------------
train = pd.read_parquet('../data/model/final_train.parquet')
val   = pd.read_parquet('../data/model/final_val.parquet')
test  = pd.read_parquet('../data/model/final_test.parquet')

train = train.sort_values(["tic", "Date"]).reset_index(drop=True)
val   = val.sort_values(["tic", "Date"]).reset_index(drop=True)
test  = test.sort_values(["tic", "Date"]).reset_index(drop=True)

news_features = ["mean_sentiment", "max_sentiment", "min_sentiment",
                 "sum_sentiment", "news_count"]

emb_cols = [c for c in train.columns if c.startswith("pca_emb_")]
print("Embedding columns:", len(emb_cols))

# ----------------------------------------
# 0.1 Add ticker ID as numeric feature
# ----------------------------------------
all_tics = sorted(pd.concat([train["tic"], val["tic"], test["tic"]]).unique())
tic2id = {tic: i for i, tic in enumerate(all_tics)}
print("Num tickers:", len(tic2id))

for df in (train, val, test):
    df["tic_id"] = df["tic"].map(tic2id).astype("int32")

# ----------------------------------------
# 1. Target
# ----------------------------------------
def add_target(df):
    df = df.sort_values(["tic", "Date"]).copy()
    grp_close = df.groupby("tic")["Close"]
    df["ret_1d_raw"] = grp_close.pct_change(1)
    df["target_1d"] = df.groupby("tic")["ret_1d_raw"].shift(-1)
    return df

train = add_target(train)
val   = add_target(val)
test  = add_target(test)

# ----------------------------------------
# 2. Price features
# ----------------------------------------
def add_price_features(df):
    df = df.sort_values(["tic","Date"]).copy()

    grp_close = df.groupby("tic")["Close"]
    grp_ret   = df.groupby("tic")["ret_1d_raw"]
    grp_vol   = df.groupby("tic")["Volume"]

    # Price momentum
    df["ret_1d_lag"] = grp_close.pct_change(1)
    df["ret_2d"]     = grp_close.pct_change(2)
    df["ret_3d"]     = grp_close.pct_change(3)
    df["ret_5d"]     = grp_close.pct_change(5)
    df["ret_10d"]    = grp_close.pct_change(10)

    # Intraday pieces
    prev_close = grp_close.shift(1)
    df["overnight_ret"] = df["Open"] / prev_close - 1
    df["intraday_ret"]  = df["Close"] / df["Open"] - 1

    # RSI
    delta = grp_close.diff()
    gain  = delta.clip(lower=0)
    loss  = -delta.clip(upper=0)
    roll_up = gain.groupby(df["tic"]).rolling(14).mean().reset_index(0, drop=True)
    roll_down = loss.groupby(df["tic"]).rolling(14).mean().reset_index(0, drop=True)
    df["RSI"] = 100 - (100 / (1 + roll_up / roll_down))

    # Volatility
    df["vol_3d"]  = grp_ret.rolling(3).std().reset_index(0, drop=True)
    df["vol_5d"]  = grp_ret.rolling(5).std().reset_index(0, drop=True)
    df["vol_10d"] = grp_ret.rolling(10).std().reset_index(0, drop=True)

    # High-Low range volatility
    df["range_vol"] = np.log(df["High"] / df["Low"]) ** 2

    # Liquidity
    mean_vol = grp_vol.transform("mean")
    std_vol  = grp_vol.transform("std")
    df["turnover"] = df["Volume"] / mean_vol
    df["vol_z"]    = (df["Volume"] - mean_vol) / std_vol
    df["amihud"]   = (np.abs(df["ret_1d_raw"]) / df["Volume"]).replace(np.inf, np.nan)

    df = df.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return df

train = add_price_features(train)
val   = add_price_features(val)
test  = add_price_features(test)

# ----------------------------------------
# 3. News features
# ----------------------------------------
def add_news_features(df):
    df = df.sort_values(["tic","Date"]).copy()
    grp = df.groupby("tic")

    for f in news_features:
        # lagged sentiment
        df[f + "_lag1"] = grp[f].shift(1)

        # rolling windows ending at t (leak-free)
        df[f + "_roll3"] = grp[f].rolling(3).mean().reset_index(0, drop=True)
        df[f + "_roll5"] = grp[f].rolling(5).mean().reset_index(0, drop=True)

        # sentiment volatility
        df[f + "_vol5"] = grp[f].rolling(5).std().reset_index(0, drop=True)

        # sentiment shocks
        df[f + "_shock"] = df[f] - df[f + "_lag1"]

        # sentiment surprise vs short-term mean
        df[f + "_surprise3"] = df[f] - df[f + "_roll3"]

    return df.fillna(0.0)

train = add_news_features(train)
val   = add_news_features(val)
test  = add_news_features(test)

# ----------------------------------------
# 4. Drop NaN target
# ----------------------------------------
train = train[~train["target_1d"].isna()].copy()
val   = val[~val["target_1d"].isna()].copy()
test  = test[~test["target_1d"].isna()].copy()

y_train = train["target_1d"].values
y_val   = val["target_1d"].values
y_test  = test["target_1d"].values

# ----------------------------------------
# 5. Feature sets (now WITH ticker)
# ----------------------------------------
price_features = [
    "ret_1d_lag", "ret_2d", "ret_3d", "ret_5d", "ret_10d",
    "overnight_ret", "intraday_ret",
    "RSI",
    "vol_3d", "vol_5d", "vol_10d",
    "range_vol",
    "turnover", "vol_z", "amihud",
    "tic_id",                    # <-- ticker as numeric feature
]

news_momentum_features = []
for f in news_features:
    news_momentum_features += [
        f,                  # same-day sentiment
        f + "_lag1",
        f + "_roll3",
        f + "_roll5",
        f + "_vol5",
        f + "_shock",
        f + "_surprise3",
    ]

embedding_features = emb_cols

all_features = price_features + news_momentum_features
all_features_with_emb = price_features + news_momentum_features + embedding_features

# ----------------------------------------
# 6. XGBoost params (base)
# ----------------------------------------
params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "tree_method": "hist",
    "device": "cuda",
    "seed": 42,
}

# if no tuning, use these best params
if 'best_params_price' not in globals() or best_params_price is None:
    best_params_price = {
        'max_depth': 4,
        'min_child_weight': 7.3505557352123505,
        'eta': 0.010017838116843746,
        'subsample': 0.9777856924636916,
        'colsample_bytree': 0.6303799594249203,
        'lambda': 0.008612782635599168,
        'alpha': 3.7035837717863744
    }
if 'best_params_news' not in globals() or best_params_news is None:
    best_params_news = {
        'max_depth': 3,
        'min_child_weight': 5.19102835385981,
        'eta': 0.013971470063586327,
        'subsample': 0.825393866505445,
        'colsample_bytree': 0.6343691159723056,
        'lambda': 1.3872924158083205e-05,
        'alpha': 4.601521799703412
    }
if 'best_params_emb' not in globals() or best_params_emb is None:
    best_params_emb = {
        "max_depth": 6,
        "min_child_weight": 7.0,
        "eta": 0.015,
        "subsample": 0.9,
        "colsample_bytree": 0.7,
        "lambda": 2.0,
        "alpha": 0.1,
    }

# ----------------------------------------
# 7. Training helper
# ----------------------------------------
def train_and_eval(name, features):
    print(f"\n===== {name} =====")

    X_train = train[features].values
    X_val   = val[features].values
    X_test  = test[features].values

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval   = xgb.DMatrix(X_val, label=y_val)
    dtest  = xgb.DMatrix(X_test, label=y_test)

    if name == "PRICE ONLY":
        tuned = best_params_price
    elif name == "PRICE + NEWS":
        tuned = best_params_news
    else:  # embeddings version
        tuned = best_params_emb

    model = xgb.train(
        params={**params, **tuned},
        dtrain=dtrain,
        evals=[(dtrain, "train"), (dval, "val")],
        num_boost_round=2000,
        early_stopping_rounds=50,
        verbose_eval=50,
    )

    preds = model.predict(dtest)

    da   = (np.sign(preds) == np.sign(y_test)).mean()
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    ic   = spearmanr(preds, y_test).correlation

    print(f"DA:   {da:.4f}")
    print(f"RMSE: {rmse:.6f}")
    print(f"IC:   {ic:.4f}")

    return {"DA": da, "RMSE": rmse, "IC": ic}, model

# ----------------------------------------
# 8. Run all three configs (all with tic_id)
# ----------------------------------------
metrics_price, model_price = train_and_eval("PRICE ONLY", price_features)
metrics_multi, model_multi = train_and_eval("PRICE + NEWS", all_features)
metrics_emb, model_emb = train_and_eval("PRICE + NEWS + EMBEDDINGS",
                                        all_features_with_emb)

print("\n=== SUMMARY ===")
print("Price only:", metrics_price)
print("Price + news:", metrics_multi)
print("Price + news + embeddings:", metrics_emb)
